In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/2025-sep-dl-gen-ai-project/sample_submission.csv
/kaggle/input/2025-sep-dl-gen-ai-project/train.csv
/kaggle/input/2025-sep-dl-gen-ai-project/test.csv


In [2]:
# Cell 2 - Imports and seeds
import os, random, json, math, gc, time, re, traceback
from pathlib import Path
import numpy as np, pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_recall_curve, average_precision_score

# transformers & optimizer
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW

# wandb (optional)
try:
    import wandb
    WANDB_INSTALLED = True
except Exception:
    WANDB_INSTALLED = False

# Paths - Kaggle input
TRAIN_PATH = Path('/kaggle/input/2025-sep-dl-gen-ai-project/train.csv')
TEST_PATH  = Path('/kaggle/input/2025-sep-dl-gen-ai-project/test.csv')
SAMPLE_PATH = Path('/kaggle/input/2025-sep-dl-gen-ai-project/sample_submission.csv')


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

Device: cuda


In [3]:
# Cell 1 - Config
METHOD = 'transformer_roberta_kfold'  # options: transformer_roberta_single, transformer_roberta_kfold, tfidf_lr, tfidf_lr_tuned, simple_embed, random
SEED = 42

# seed
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

MAX_TRAIN_ROWS = None
TRAIN_EPOCHS_LIGHT = 5   # keep small for quick runs; increase for final runs
BATCH_SIZE_LIGHT = 32
MAX_LENGTH_TRANSFORMER = 128

USE_CLEAN_TEXT = True
SHOW_PR_CURVES = True
TUNE_THRESHOLDS_GRID = (0.01, 0.99, 99)

LABELS = ['anger', 'fear', 'joy', 'sadness', 'surprise']
REQUIRED_COLUMNS = ['id'] + LABELS

TFIDF_MAX_FEATURES = 120000
WORD_NGRAMS = (1, 2)
CHAR_NGRAMS = (2, 6)
CHAR_ANALYZER = 'char_wb'
MIN_DF = 2
USE_IDF = True
SUBLINEAR_TF = True
LOWERCASE = True

LR_C = 2.0
LR_SOLVER = 'saga'
LR_PENALTY = 'elasticnet'
LR_L1_RATIO = 0.25
LR_MAX_ITER = 4000
THRESHOLD = 0.5

DT_MAX_DEPTH = 6
DT_RANDOM_STATE = 42
TEST_SIZE = 0.2

EMBED_DIM = 64
LSTM_HIDDEN = 64
LSTM_DROPOUT = 0.2
MAX_LEN_SEQ = 64

TRANSFORMER_NAME_ROBERTA = 'roberta-base'
TRANSFORMER_NAME_BERT = 'bert-base-uncased'

USE_WANDB = True
WANDB_PROJECT = '21f1004482-t32025'
WANDB_RUN_NAME = f'kaggle_{METHOD}'

print({'method': METHOD, 'epochs_light': TRAIN_EPOCHS_LIGHT})


{'method': 'transformer_roberta_kfold', 'epochs_light': 5}


In [4]:
# Cell 3 - Load data & quick EDA
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
if SAMPLE_PATH.exists():
    sample = pd.read_csv(SAMPLE_PATH)
else:
    sample = pd.DataFrame(columns=REQUIRED_COLUMNS)

print("Train shape:", train.shape, "Test shape:", test.shape)
train['text'] = train['text'].fillna('').astype(str)
test['text'] = test['text'].fillna('').astype(str)

if MAX_TRAIN_ROWS:
    train = train.sample(MAX_TRAIN_ROWS, random_state=SEED)

priors = {lbl: float(train[lbl].mean()) for lbl in LABELS}
print("Label priors:", priors)

train['text_len'] = train['text'].str.len()
print("Text length mean, std:", train['text_len'].mean(), train['text_len'].std())


Train shape: (6827, 8) Test shape: (1707, 2)
Label priors: {'anger': 0.11835359601581955, 'fear': 0.5654020799765637, 'joy': 0.24315218983448073, 'sadness': 0.3180020506811191, 'surprise': 0.2928079683609199}
Text length mean, std: 79.43078951223085 57.3973970218847


In [5]:
# Cell 4 - Cleaning
if USE_CLEAN_TEXT:
    PAT_URL = re.compile(r'http\S+|www\S+')
    PAT_USER = re.compile(r'@[A-Za-z0-9_]+')
    PAT_HASH = re.compile(r'#[A-Za-z0-9_]+')
    PAT_NON_ALNUM = re.compile(r'[^a-z0-9\s]')
    def clean_text(s: str) -> str:
        s = s.lower()
        s = PAT_URL.sub(' ', s)
        s = PAT_USER.sub(' ', s)
        s = PAT_HASH.sub(' ', s)
        s = PAT_NON_ALNUM.sub(' ', s)
        s = re.sub(r'\s+', ' ', s).strip()
        return s
    train['text_clean'] = train['text'].apply(clean_text)
    test['text_clean'] = test['text'].apply(clean_text)
else:
    train['text_clean'] = train['text']
    test['text_clean'] = test['text']
print("Sample cleaned text:", train['text_clean'].iloc[0][:200])


Sample cleaned text: the dentist that did the work apparently did a lousy job as in just a few years my teeth decayed under the crowns so i had no choice but to get partials


In [6]:
# Cell 5 - Utilities
def macro_f1(y_true, y_pred):
    return float(np.mean([f1_score(y_true[:,i], y_pred[:,i], zero_division=0) for i in range(y_true.shape[1])]))

def sweep_thresholds(y_true: np.ndarray, probs: np.ndarray, grid=None):
    if grid is None:
        grid = np.linspace(*TUNE_THRESHOLDS_GRID)
    best = []
    for i in range(probs.shape[1]):
        best_thr, best_f1 = 0.5, -1.0
        for thr in grid:
            pred = (probs[:, i] >= thr).astype(int)
            f1 = f1_score(y_true[:, i], pred, zero_division=0)
            if f1 > best_f1:
                best_f1, best_thr = f1, thr
        best.append(best_thr)
    return np.array(best)

def plot_pr_curves(y_true, probs, labels=LABELS, save_path=None):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8,6))
    for i,lbl in enumerate(labels):
        prec, rec, _ = precision_recall_curve(y_true[:, i], probs[:, i])
        ap = average_precision_score(y_true[:, i], probs[:, i])
        ax.plot(rec, prec, label=f"{lbl} (AP={ap:.3f})")
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision'); ax.set_title('PR curves')
    ax.legend()
    if save_path:
        fig.savefig(save_path, dpi=120)
        print("Saved PR curves to", save_path)
    else:
        plt.show()
    plt.close(fig)


In [7]:
# Cell 6 - Random prior baseline
def random_prior_submission(train_df, test_df, seed=SEED):
    rng = np.random.default_rng(seed)
    priors_local = {lbl: float(train_df[lbl].mean()) for lbl in LABELS}
    out = pd.DataFrame({'id': test_df['id']})
    for lbl in LABELS:
        p = priors_local[lbl]
        out[lbl] = rng.binomial(1, p, size=len(test_df)).astype(int)
    return out

baseline_sub = random_prior_submission(train, test)
baseline_sub.to_csv("submission_baseline_random.csv", index=False)
print("Saved submission_baseline_random.csv")

Saved submission_baseline_random.csv


In [8]:
# Cell 7 - TF-IDF + Logistic Regression (optional)
from scipy.sparse import hstack
def tfidf_lr_train_predict(train_df, test_df, tune_thresholds=False):
    text_col = 'text_clean' if 'text_clean' in train_df.columns else 'text'
    X_text = train_df[text_col].astype(str).values
    y = train_df[LABELS].astype(int).values

    word_vec = TfidfVectorizer(max_features=TFIDF_MAX_FEATURES, ngram_range=WORD_NGRAMS, min_df=MIN_DF,
                               use_idf=USE_IDF, sublinear_tf=SUBLINEAR_TF, lowercase=LOWERCASE, analyzer='word')
    char_vec = TfidfVectorizer(max_features=TFIDF_MAX_FEATURES, ngram_range=CHAR_NGRAMS, min_df=MIN_DF,
                               use_idf=USE_IDF, sublinear_tf=SUBLINEAR_TF, lowercase=LOWERCASE, analyzer=CHAR_ANALYZER)
    Xw = word_vec.fit_transform(X_text)
    Xc = char_vec.fit_transform(X_text)
    X = hstack([Xw, Xc])

    X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, shuffle=True)
    base_lr = LogisticRegression(C=LR_C, solver=LR_SOLVER, max_iter=LR_MAX_ITER,
                                 penalty=LR_PENALTY, l1_ratio=LR_L1_RATIO, class_weight='balanced', n_jobs=-1)
    clf = OneVsRestClassifier(base_lr, n_jobs=1)
    clf.fit(X_tr, y_tr)

    try:
        va_probs = clf.predict_proba(X_va)
    except Exception:
        from scipy.special import expit
        va_scores = clf.decision_function(X_va)
        va_probs = expit(va_scores)

    val_preds_flat = (va_probs >= THRESHOLD).astype(int)
    val_macro_flat = f1_score(y_va, val_preds_flat, average='macro', zero_division=0)

    thresholds = None
    val_macro_tuned = None
    if tune_thresholds:
        grid = np.linspace(*TUNE_THRESHOLDS_GRID)
        thresholds = sweep_thresholds(y_va, va_probs, grid=grid)
        val_preds_tuned = (va_probs >= thresholds).astype(int)
        val_macro_tuned = f1_score(y_va, val_preds_tuned, average='macro', zero_division=0)
        if SHOW_PR_CURVES:
            plot_pr_curves(y_va, va_probs, save_path="tfidf_val_pr_curves.png")

    Xt_w = word_vec.transform(test_df[text_col].astype(str))
    Xt_c = char_vec.transform(test_df[text_col].astype(str))
    Xt = hstack([Xt_w, Xt_c])
    try:
        test_probs = clf.predict_proba(Xt)
    except Exception:
        from scipy.special import expit
        test_scores = clf.decision_function(Xt)
        test_probs = expit(test_scores)

    if thresholds is None:
        test_preds = (test_probs >= THRESHOLD).astype(int)
    else:
        test_preds = (test_probs >= thresholds).astype(int)

    sub = pd.DataFrame({'id': test_df['id']})
    for i,lbl in enumerate(LABELS):
        sub[lbl] = test_preds[:, i].astype(int)

    metrics = {'val_macro_flat': float(val_macro_flat)}
    if val_macro_tuned is not None:
        metrics['val_macro_tuned'] = float(val_macro_tuned)
        metrics['thresholds'] = thresholds.tolist()

    return sub, metrics, clf, word_vec, char_vec

if METHOD in ('tfidf_lr','tfidf_lr_tuned'):
    tune = METHOD == 'tfidf_lr_tuned'
    sub_tfidf, metrics_tfidf, clf_obj, word_vec, char_vec = tfidf_lr_train_predict(train, test, tune_thresholds=tune)
    sub_tfidf.to_csv("submission_tfidf_lr.csv", index=False)
    print("Saved submission_tfidf_lr.csv — metrics:", metrics_tfidf)

In [9]:
# Cell 8 - SimpleEmbed (from-scratch)
class SimpleEmbedDataset(Dataset):
    def __init__(self, texts, labels=None, stoi=None, max_len=MAX_LEN_SEQ):
        self.texts = texts
        self.labels = labels
        self.stoi = stoi
        self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        toks = self.texts[idx].lower().split()
        ids = [self.stoi.get(t, 1) for t in toks][:self.max_len]
        if len(ids) < self.max_len: ids += [0]*(self.max_len-len(ids))
        item = {'input_ids': torch.tensor(ids, dtype=torch.long)}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float32)
        return item

class SimpleEmbedModel(nn.Module):
    def __init__(self, vocab_size, emb_dim=EMBED_DIM, num_labels=len(LABELS)):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.fc = nn.Linear(emb_dim, num_labels)
    def forward(self, input_ids):
        x = self.emb(input_ids)
        mask = (input_ids != 0).unsqueeze(-1).float()
        summed = (x * mask).sum(1)
        denom = mask.sum(1).clamp(min=1.0)
        pooled = summed / denom
        logits = self.fc(pooled)
        return logits

if METHOD == 'simple_embed':
    text_col = 'text_clean'
    toks = [t.lower().split() for t in train[text_col].tolist()]
    from collections import Counter
    cnt = Counter()
    for row in toks: cnt.update(row)
    itos = ['<pad>','<unk>'] + [w for w,_ in cnt.most_common(20000) if w not in ('<pad','<unk>')]
    stoi = {w:i for i,w in enumerate(itos)}
    X = train[text_col].tolist()
    y = train[LABELS].values
    X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, shuffle=True)
    tr_ds = SimpleEmbedDataset(X_tr, y_tr, stoi, max_len=MAX_LEN_SEQ)
    va_ds = SimpleEmbedDataset(X_va, y_va, stoi, max_len=MAX_LEN_SEQ)
    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE_LIGHT, shuffle=True, num_workers=2)
    va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE_LIGHT, shuffle=False, num_workers=2)

    model = SimpleEmbedModel(vocab_size=len(itos), emb_dim=EMBED_DIM).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=2e-3)
    crit = nn.BCEWithLogitsLoss()
    best_f1 = -1.0
    for ep in range(1, TRAIN_EPOCHS_LIGHT+1):
        model.train()
        run_loss = 0.0
        for b in tr_loader:
            input_ids = b['input_ids'].to(DEVICE)
            labels = b['labels'].to(DEVICE)
            logits = model(input_ids)
            loss = crit(logits, labels)
            opt.zero_grad(); loss.backward(); opt.step()
            run_loss += loss.item() * input_ids.size(0)
        avg_train_loss = run_loss / len(tr_loader.dataset)
        model.eval()
        preds=[]
        trues=[]
        with torch.no_grad():
            for b in va_loader:
                input_ids = b['input_ids'].to(DEVICE)
                labels = b['labels'].cpu().numpy()
                logits = model(input_ids).cpu().numpy()
                probs = 1/(1+np.exp(-logits))
                preds.append(probs); trues.append(labels)
        preds = np.vstack(preds); trues = np.vstack(trues)
        val_pred_flat = (preds >= 0.5).astype(int)
        val_f1_flat = f1_score(trues, val_pred_flat, average='macro', zero_division=0)
        print(f"Epoch {ep} train_loss={avg_train_loss:.4f} val_macro_f1_flat={val_f1_flat:.4f}")
        if val_f1_flat > best_f1:
            best_f1 = val_f1_flat
            torch.save(model.state_dict(), "best_simple_embed.pth")
    test_ds = SimpleEmbedDataset(test[text_col].tolist(), labels=None, stoi=stoi, max_len=MAX_LEN_SEQ)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE_LIGHT, shuffle=False, num_workers=2)
    model.load_state_dict(torch.load("best_simple_embed.pth"))
    model.eval()
    all_probs=[]
    with torch.no_grad():
        for b in test_loader:
            input_ids = b['input_ids'].to(DEVICE)
            logits = model(input_ids)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.append(probs)
    all_probs = np.vstack(all_probs)
    preds = (all_probs >= 0.5).astype(int)
    sub = pd.DataFrame({'id': test['id']})
    for i,lbl in enumerate(LABELS):
        sub[lbl] = preds[:,i].astype(int)
    sub.to_csv("submission_simple_embed.csv", index=False)
    print("Saved submission_simple_embed.csv — best_val_f1:", best_f1)

In [10]:
# Cell 9 - Safe HF loader (used by both single-run and k-fold)
import importlib
# monkey-patch HF repo listing to avoid 404 on templates in some HF versions
try:
    import transformers.utils.hub as _t_hub
    def _noop(*args, **kwargs): return []
    if hasattr(_t_hub, "list_repo_templates"): _t_hub.list_repo_templates = _noop
    if hasattr(_t_hub, "list_repo_tree"): _t_hub.list_repo_tree = _noop
    if hasattr(_t_hub, "list_repo_files"): _t_hub.list_repo_files = _noop
    import huggingface_hub.hf_api as _hf_api
    if hasattr(_hf_api, "HfApi"):
        _hf_api.HfApi.list_repo_files = _noop
    _hf_api.list_repo_files = _noop
except Exception:
    pass

def safe_load_tokenizer_and_model(requested_model_name, num_labels=len(LABELS), cache_dir=None):
    CANDIDATES = [
        requested_model_name,
        "roberta-base",
        "distilroberta-base",
        "bert-base-uncased",
        "distilbert-base-uncased"
    ]
    last_exc = None
    for nm in CANDIDATES:
        try:
            print(f"Attempting tokenizer: {nm}")
            tok = AutoTokenizer.from_pretrained(nm, use_fast=True, trust_remote_code=False, cache_dir=cache_dir)
            print(f"Tokenizer OK: {nm} — attempting model")
            mdl = AutoModelForSequenceClassification.from_pretrained(
                nm,
                num_labels=num_labels,
                problem_type="multi_label_classification",
                trust_remote_code=False,
                cache_dir=cache_dir
            )
            print(f"Model OK: {nm}")
            return tok, mdl, nm
        except Exception as e:
            last_exc = e
            print(f"Failed to load {nm}: {str(e)[:200]}")
            continue
    raise RuntimeError(f"All tokenizer/model fallbacks failed. Last error: {last_exc}")

In [11]:
# Cell 10 - Single-run RoBERTa
def run_roberta_single():
    print("\n===== Running Single-Run RoBERTa =====")
    text_col = 'text_clean' if USE_CLEAN_TEXT else 'text'
    X = train[text_col].tolist()
    y = train[LABELS].values
    X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, shuffle=True)

    tokenizer, model, used_name = safe_load_tokenizer_and_model(TRANSFORMER_NAME_ROBERTA)
    model.to(DEVICE)

    class TfDataset(torch.utils.data.Dataset):
        def __init__(self, texts, labels=None):
            self.texts = texts; self.labels = labels
        def __len__(self): return len(self.texts)
        def __getitem__(self, idx):
            enc = tokenizer(self.texts[idx], truncation=True, padding='max_length',
                            max_length=MAX_LENGTH_TRANSFORMER, return_tensors='pt')
            item = {k:v.squeeze(0) for k,v in enc.items()}
            if self.labels is not None:
                item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float32)
            return item

    tr_ds = TfDataset(X_tr, y_tr); va_ds = TfDataset(X_va, y_va)
    dl_tr = DataLoader(tr_ds, batch_size=BATCH_SIZE_LIGHT, shuffle=True)
    dl_va = DataLoader(va_ds, batch_size=BATCH_SIZE_LIGHT, shuffle=False)

    optimizer = AdamW(model.parameters(), lr=2e-5)
    total_steps = max(1, len(dl_tr) * TRAIN_EPOCHS_LIGHT)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps)

    best_f1 = -1; best_state = None; best_thr = np.array([0.5]*len(LABELS))
    for ep in range(1, TRAIN_EPOCHS_LIGHT+1):
        model.train(); run_loss = 0.0
        for batch in dl_tr:
            labels = batch.pop('labels').to(DEVICE)
            batch = {k:v.to(DEVICE) for k,v in batch.items()}
            out = model(**batch, labels=labels)
            loss = out.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()
            run_loss += loss.item() * labels.size(0)
        avg_loss = run_loss / len(dl_tr.dataset)

        # validation
        model.eval(); val_probs=[]; val_trues=[]
        with torch.no_grad():
            for batch in dl_va:
                labels = batch.pop('labels').cpu().numpy()
                batch = {k:v.to(DEVICE) for k,v in batch.items()}
                logits = model(**batch).logits
                probs = torch.sigmoid(logits).cpu().numpy()
                val_probs.append(probs); val_trues.append(labels)
        val_probs = np.vstack(val_probs); val_trues = np.vstack(val_trues)
        thr = sweep_thresholds(val_trues, val_probs)
        preds = (val_probs >= thr).astype(int)
        val_f1 = f1_score(val_trues, preds, average='macro', zero_division=0)
        print(f"Epoch {ep} | loss={avg_loss:.4f} | val_macro_f1={val_f1:.4f}")
        if val_f1 > best_f1:
            best_f1 = val_f1; best_state = {k:v.cpu() for k,v in model.state_dict().items()}; best_thr = thr.copy()

    model.load_state_dict(best_state); model.eval()

    # test inference
    test_texts = test[text_col].tolist()
    all_probs=[]
    for i in range(0, len(test_texts), 256):
        enc = tokenizer(test_texts[i:i+256], truncation=True, padding='max_length', max_length=MAX_LENGTH_TRANSFORMER, return_tensors='pt')
        enc = {k:v.to(DEVICE) for k,v in enc.items()}
        with torch.no_grad():
            logits = model(**enc).logits
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.append(probs)
    all_probs = np.vstack(all_probs)
    preds = (all_probs >= best_thr).astype(int)

    sub = pd.DataFrame({'id': test['id']})
    for i,lbl in enumerate(LABELS):
        sub[lbl] = preds[:,i].astype(int)
    sub.to_csv('submission_transformer_roberta_single.csv', index=False)
    print('Saved submission_transformer_roberta_single.csv — best_val_f1:', best_f1)

# run if selected
if METHOD == 'transformer_roberta_single':
    run_roberta_single()

In [12]:
# Cell 11 - 5-Fold RoBERTa (k-fold ensemble) - RUNS ONLY if METHOD == 'transformer_roberta_kfold'
if METHOD == 'transformer_roberta_kfold':
    from sklearn.model_selection import StratifiedKFold

    # prepare stratify
    train_strat = train[LABELS].astype(str).agg('-'.join, axis=1)
    FOLDS = 5
    skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

    all_oof_probs = []
    all_oof_trues = []
    test_fold_probs = []
    thresholds_per_fold = []

    def train_one_fold(fold_id, tr_idx, va_idx):
        text_col = 'text_clean' if USE_CLEAN_TEXT else 'text'
        X_tr = train.iloc[tr_idx][text_col].tolist()
        X_va = train.iloc[va_idx][text_col].tolist()
        y_tr = train.iloc[tr_idx][LABELS].values
        y_va = train.iloc[va_idx][LABELS].values

        tokenizer, model, used_name = safe_load_tokenizer_and_model(TRANSFORMER_NAME_ROBERTA, num_labels=len(LABELS))
        model.to(DEVICE)

        class SeqDataset(torch.utils.data.Dataset):
            def __init__(self, texts, labels=None, tokenizer=None, max_len=MAX_LENGTH_TRANSFORMER):
                self.texts = texts; self.labels = labels; self.tokenizer = tokenizer; self.max_len = max_len
            def __len__(self): return len(self.texts)
            def __getitem__(self, idx):
                enc = tokenizer(self.texts[idx], truncation=True, padding='max_length', max_length=self.max_len, return_tensors='pt')
                item = {k:v.squeeze(0) for k,v in enc.items()}
                if self.labels is not None:
                    item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float32)
                return item

        tr_ds = SeqDataset(X_tr, y_tr)
        va_ds = SeqDataset(X_va, y_va)
        dl_tr = DataLoader(tr_ds, batch_size=BATCH_SIZE_LIGHT, shuffle=True, num_workers=2)
        dl_va = DataLoader(va_ds, batch_size=BATCH_SIZE_LIGHT, shuffle=False, num_workers=2)

        optimizer = AdamW(model.parameters(), lr=2e-5)
        steps = len(dl_tr) * TRAIN_EPOCHS_LIGHT
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*steps), num_training_steps=steps)

        best_f1 = -1; best_state = None; best_thr = np.array([0.5]*len(LABELS))

        for ep in range(1, TRAIN_EPOCHS_LIGHT+1):
            model.train(); run_loss = 0.0
            for batch in tqdm(dl_tr, desc=f"[Fold {fold_id}] Epoch {ep}"):
                labels = batch.pop('labels').to(DEVICE)
                batch = {k:v.to(DEVICE) for k,v in batch.items()}
                out = model(**batch, labels=labels)
                loss = out.loss
                optimizer.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); scheduler.step()
                run_loss += loss.item() * labels.size(0)
            avg_loss = run_loss / max(1, len(dl_tr.dataset))

            model.eval(); val_probs = []; val_trues = []
            with torch.no_grad():
                for batch in dl_va:
                    labels = batch.pop('labels').cpu().numpy()
                    batch = {k:v.to(DEVICE) for k,v in batch.items()}
                    logits = model(**batch).logits
                    probs = torch.sigmoid(logits).cpu().numpy()
                    val_probs.append(probs); val_trues.append(labels)
            val_probs = np.vstack(val_probs); val_trues = np.vstack(val_trues)
            thr = sweep_thresholds(val_trues, val_probs)
            val_preds = (val_probs >= thr).astype(int)
            val_f1 = f1_score(val_trues, val_preds, average='macro', zero_division=0)
            print(f"Fold {fold_id} Epoch {ep} val_f1={val_f1:.4f}")
            if val_f1 > best_f1:
                best_f1 = val_f1; best_state = {k:v.cpu() for k,v in model.state_dict().items()}; best_thr = thr.copy()

        print(f"Fold {fold_id} BEST F1: {best_f1:.4f}")
        return best_state, best_thr, (val_trues, val_probs), tokenizer, used_name

    fold_num = 1
    for tr_idx, va_idx in skf.split(train, train_strat):
        print(f"\n============= FOLD {fold_num} =============")
        best_state, best_thr, (va_trues, va_probs), tokenizer_used, name_used = train_one_fold(fold_num, tr_idx, va_idx)
        thresholds_per_fold.append(best_thr)
        all_oof_probs.append(va_probs); all_oof_trues.append(va_trues)

        # load best state for test inference
        model = AutoModelForSequenceClassification.from_pretrained(name_used, num_labels=len(LABELS), problem_type='multi_label_classification')
        model.load_state_dict(best_state); model.to(DEVICE); model.eval()

        # test inference for fold
        text_col = 'text_clean' if USE_CLEAN_TEXT else 'text'
        test_texts = test[text_col].tolist()
        fold_probs = []
        for i in range(0, len(test_texts), 256):
            batch_texts = test_texts[i:i+256]
            enc = tokenizer_used(batch_texts, truncation=True, padding='max_length', max_length=MAX_LENGTH_TRANSFORMER, return_tensors='pt')
            enc = {k:v.to(DEVICE) for k,v in enc.items()}
            with torch.no_grad():
                logits = model(**enc).logits
                probs = torch.sigmoid(logits).cpu().numpy()
                fold_probs.append(probs)
        test_fold_probs.append(np.vstack(fold_probs))
        fold_num += 1
        gc.collect(); torch.cuda.empty_cache()

    # OOF tuning & ensemble
    oof_true = np.vstack(all_oof_trues); oof_prob = np.vstack(all_oof_probs)
    global_thr = sweep_thresholds(oof_true, oof_prob)
    print('Global thresholds:', dict(zip(LABELS, global_thr.round(3))))
    test_prob = np.mean(np.stack(test_fold_probs, axis=0), axis=0)
    test_pred = (test_prob >= global_thr).astype(int)
    submission = pd.DataFrame({'id': test['id']})
    for i,lbl in enumerate(LABELS):
        submission[lbl] = test_pred[:,i].astype(int)
    submission.to_csv('submission_5fold_roberta.csv', index=False)
    print('Saved submission_5fold_roberta.csv')


============= FOLD 1 =============
Attempting tokenizer: roberta-base


/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_split.py:700: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

...

[Fold 5] Epoch 5:   0%|          | 0/171 [00:00<?, ?it/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Fold 5 Epoch 5 val_f1=0.8199
Fold 5 BEST F1: 0.8199


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Global thresholds: {'anger': 0.32, 'fear': 0.52, 'joy': 0.44, 'sadness': 0.51, 'surprise': 0.41}
Saved submission_5fold_roberta.csv


In [13]:
# Cell 12 - Optional W&B finalize
if USE_WANDB:
    if WANDB_INSTALLED:
        try:
            if os.environ.get('WANDB_API_KEY') is None:
                try:
                    from kaggle_secrets import UserSecretsClient
                    key = UserSecretsClient().get_secret("WANDB_API_KEY")
                    os.environ['WANDB_API_KEY'] = key
                except Exception:
                    pass
            wandb.init(project=WANDB_PROJECT, name=WANDB_RUN_NAME, reinit=True)
            wandb.log({"note": "Experiment finished"})
            wandb.finish()
            print("W&B done (if API key provided).")
        except Exception as e:
            print("W&B skipped:", e)
    else:
        print("wandb not installed; set USE_WANDB=False to silence.")

wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.21.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20251127_133201-y68t4okt
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run kaggle_transformer_roberta_kfold
wandb: ⭐️ View project at https://wandb.ai/21f1004482-dl-genai-project/21f1004482-t32025
wandb: 🚀 View run at https://wandb.ai/21f1004482-dl-genai-project/21f1004482-t32025/runs/y68t4okt
wandb:                                                                                
wandb: 
wandb: Run summary:
wandb: note Experiment finished
wandb: 
wandb: 🚀 View run kaggle_transformer_roberta_kfold at: https://wandb.ai/21f1004482-dl-genai-project/21f1004482-t32025/runs/y68t4okt
wandb: ⭐️ View project at: https://wandb.ai/21f1004482-dl-genai-project/21f1004482-t32025
wandb: 

W&B done (if API key provided).
